In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("bronze")
    .master("local[*]")
    .config("spark.eventLog.enabled", "false")
    .getOrCreate()
)

In [3]:
import os

PASSWORD = os.environ.get("SQL_PASSWORD")

print("Password loaded:", PASSWORD is not None)

Password loaded: True


In [4]:
print("PASSWORD variable matches .env:", PASSWORD == "1234")
print("SQL_PASSWORD variable matches .env:", os.environ.get("SQL_PASSWORD") == "1234")

PASSWORD variable matches .env: True
SQL_PASSWORD variable matches .env: True


In [5]:
dim_customer = spark.read.jdbc(
    url=(
        "jdbc:sqlserver://host.docker.internal:1433;"
        "databaseName=ContosoRetailDW;"
        "encrypt=true;"
        "trustServerCertificate=true"
    ),
    table="dbo.DimCustomer",
    properties={
        "user": "spark_reader",
        "password": PASSWORD,
        "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
    }
)


In [6]:
dim_customer.write \
    .mode("overwrite") \
    .parquet("file:///data-lake/data-lake/bronze/DimCustomer")

In [7]:
dim_product = spark.read.jdbc(
    url=(
        "jdbc:sqlserver://host.docker.internal:1433;"
        "databaseName=ContosoRetailDW;"
        "encrypt=true;"
        "trustServerCertificate=true"
    ),
    table="dbo.DimProduct",
    properties={
        "user": "spark_reader",
        "password": "1234",
        "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
    }
)


In [8]:
dim_product.write \
    .mode("overwrite") \
    .parquet("file:///data-lake/data-lake/bronze/DimProduct")

In [9]:
dim_store = spark.read.jdbc(
    url=(
        "jdbc:sqlserver://host.docker.internal:1433;"
        "databaseName=ContosoRetailDW;"
        "encrypt=true;"
        "trustServerCertificate=true"
    ),
    table="dbo.DimStore",
    properties={
        "user": "spark_reader",
        "password": "1234",
        "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
    }
)


In [10]:
dim_store.write \
    .mode("overwrite") \
    .parquet("file:///data-lake/data-lake/bronze/DimStore")

In [11]:
dim_onlinesales = spark.read.jdbc(
    url=(
        "jdbc:sqlserver://host.docker.internal:1433;"
        "databaseName=ContosoRetailDW;"
        "encrypt=true;"
        "trustServerCertificate=true"
    ),
    table="dbo.FactOnlineSales",
    column="OnlineSalesKey", lowerBound=19560484, upperBound=32188091, numPartitions=4,
    properties={
        "user": "spark_reader",
        "password": "1234",
        "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
        "fetchsize": "20000"
    }
)


In [12]:
dim_onlinesales.write \
    .mode("overwrite") \
    .parquet("file:///data-lake/data-lake/bronze/FactOnlineSales")